**Large Language Models**

_Copyright Notice_

This notebook is available for educational purposes. You may not use or distribute this notebook for commercial purposes. You may make copies of this notebook and use or distribute it for educational purposes as long as you cite the author as the source of the notebook.

Jaeyoung Chun | School of Applied Artificial Intelligence | Handong Global University

# Manual Retrieval-Augmented Generation

## Load Environment Variables

In [1]:
from dotenv import load_dotenv

In [2]:
# retrieve API key and other environment variables
load_dotenv()

True

## Embedding

In [3]:
from langchain_openai import OpenAIEmbeddings

In [4]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

### Documents

In [5]:
titles = ["살아있는 화석", "노아의 홍수", "빅뱅이론"]

In [6]:
doc1 = """
워너 박사는 '살아있는 화석(Living Fossils)'이란 무엇인지, 그리고 왜 그가 지난 14년 동안 살아있는 화석에 깊이 관심을 가지고 관련 사진들을 수집해왔는지에 대해서 설명해 주었다. ”살아있는 화석이란 현재 존재하는 생물과 비슷한 모양을 지닌 화석화된 동물 또는 식물들입니다. 나는 진화론을 시험해보기 위한 도구로서 살아있는 화석에 관심을 가지게 되었습니다.”

”생명이 어떻게 발생했는가에 대해서는 기본적으로 두 가지 모델이 있습니다. 진화론은 거의 40억년 전에 화학물질들이 우연히 응집되어 살아있는 단세포가 형성되었고, 이것이 오랜 시간에 걸쳐서 다른 살아있는 생물들로 바뀌었다는 제안하고 있습니다. 공룡에서 조류로의 변화, 또는 네발 달린 육지 포유류에서 고래로의 변화들이 진화론에서 주장하는 진화론적 변화의 예로 제시되고 있습니다. 또 다른 모델인 창조론에서는 외부의 초자연적인 존재(하나님)에 의해 모든 다양한 동식물들이 한꺼번에 창조되었고, 이 생물들이 시간이 흘러도 창조된 기본적 형태에서 거의 변하지 않았다고 제안하고 있습니다”
"""

In [7]:
doc2 = """
성경 창세기에 나와있는 노아의 홍수는 과연 어떠할까? 즉 ”노아 육백세 되던 해 이월 곧 그 달 십칠일이라 그 날에 큰 깊음의 샘들이 터지며 하늘의 창들이 열려 사십 주야를 비가 땅에 쏟아졌더라”(창세기 7:11~12)는 내용이 과거의 어느 시점에 지구상에서 일어났던 인류 역사상의 사실이었을까? 

그런데 이와 유사한 내용이 수메르의 점토판에 쓰여져 있다.

”아침에 비가 내리고 또 내렸다. 나는 밤에도 장대비가 내리는 것을 내 눈으로 보았다. 나는 머리를 들어 하늘을 바라보았는데, 그 두려움은 말할 수 없을 정도였다. 첫째 날에는 남풍이 무서운 속도로 불었다. 사람들은 전쟁이 시작된 것이 아닌가 하고 앞을 다투어 산 속으로 달아났다. 다른 사람을 구할 엄두도 못낸 채 정신없이 달아났다”

이것은 인류를 전멸시키기 위하여 신이 불러일으킨 홍수이야기다. 

거의 모든 생명을 앗아갔다고 하는 대홍수의 전설은 단지 메소포타미아의 수메르 지방뿐 아니라, 이집트의 경전, 인도 산스크리트어의 고서, 태평양 여러 민족의 민담, 우리나라 태고의 기록에서도, 남북아메리카 원주민과 유럽의 토착민들의 전설에서도 찾아볼 수 있다.

지금까지 이러한 각지에 남아있는 동서양의 홍수전설의 유사점들을 미루어볼 때, 그 근저에는 인류의 기억에 깊은 각인을 남긴 어떤 실제적 사건이 존재했었다고 생각할 수 있지 않을까? 

결론적으로 말한다면 노아와 그의 가족에서부터 시작하여 바벨탑의 혼란에 의해 노아의 후손들이 지구 전역으로 퍼져나가게 되고, 각 대륙과 섬들에 정착한 노아의 후손들이 각기 뿌리를 내린 지역의 문화나 풍토성에 영향을 받아 창세기 홍수와 방주의 기본골격은 어느 정도 유지한 채 지역과 민족성에 따라 조상으로부터 대대로 내려왔던 이야기들이 조금씩 원전(原典)에서 광이 윤색되고 변질되는 형태로 구비전승(具備傳承)되면서, 종국적으로 원래 성서에 기록되어 있는 홍수이야기와는 상당히 동떨어진 홍수전설로 남게된 것으로 추정된다. 예를 들면 태평양의 피지섬에선 창세기에 나오는 직사각형의 방주(Ark) 형태가 피지섬 원주민들에게 쉽게 이해될 수 있고 생활에 유용하게 쓰이고 가장 친숙한 카누(Canoe)의 형태로 변색되어 나타나고 있다는 점을 들 수 있다.
"""

In [8]:
doc3 = """
오랫동안 기다렸던 빅뱅이론(Big Bang theory)의 허구성을 밝힌 DVD가 출시되었다! 리얼 사이언스 라디오(Real Science Radio, RSR)에서 방송됐던, 빅뱅이론의 지지자인 이론물리학자 로렌스 크라우스와의 논쟁(debate with Lawrence Krauss) 후에, 우리는 빅뱅이론에 반대되는 과학적 증거들과, 그 이론에 기초한 우주론의 주요 특징들에 대한 첫 번째 프로그램을 방영했다. RSR의 빅뱅이론에 반대되는 증거들 목록에 오신 것을 환영한다! 밥 엔야트(Bob Enyart)와 공동 주최자인 프레드 윌리암(Fred Williams)이 작업한, 로렌스 크라우스와의 10초짜리 영상이 포함된 최신 방송을 들으려면 여기를 클릭하라. 크라우스 박사는 RSR에서 '모든 증거들은 빅뱅이론을 압도적으로 지지한다'고 말했다. 따라서 우리는 이에 대한 반응으로, 빅뱅이론이 허구임을 가리키는 압도적인 증거 목록을 작성했다. 이 목록은 많은 과학자들이 빅뱅이론에서 예상하는 것들과 일치하지 않는 것들로 평가된, 가장 간결하고, 가장 포괄적인 증거들에 대한 목록이다. 이 간단한 rsr.org/big-bang 링크를 게시하여, 친구들과 이 페이지를 알려주고, 이 요약 글의 바로 아래에 있는 놀라운 전체 목록을 확인하라...
"""

In [9]:
docs = [doc1, doc2, doc3]

In [10]:
embedding_docs = embedding_model.embed_documents(docs)

### Question

In [11]:
question = "노아의 홍수는 진짜 있었던 일인가요?"

In [12]:
embedding_question = embedding_model.embed_query(question)

## Retrieve

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def get_similarity(a, b):
    return cosine_similarity([a], [b])[0][0]

In [14]:
# find the most similar document
idx_to_use = -1
best_score = 0
for idx, embedding_doc in enumerate(embedding_docs):
    score = get_similarity(embedding_question, embedding_doc)
    print(titles[idx], score)
    if score > best_score:
        idx_to_use = idx
        best_score = score

살아있는 화석 0.2420369540642091
노아의 홍수 0.540774455461059
빅뱅이론 0.13450791625983882


In [15]:
idx_to_use, titles[idx_to_use]

(1, '노아의 홍수')

In [16]:
context = docs[idx_to_use]

## Augment

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage

In [18]:
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(
            content="기독교 세계관을 가지고 답해줘."
        ),
        HumanMessagePromptTemplate.from_template(
            """
            {question}
            아래의 문맥에 기반하여 답해주세요.
            {context}
            """
        )
    ]
)

In [19]:
message = chat_template.format_messages(
    question=question,
    context=context
)

## Generate

In [20]:
from langchain_openai import ChatOpenAI

In [21]:
model = ChatOpenAI(
    model_name="gpt-4o-mini",
)

In [22]:
for chunk in model.stream(message):
    print(chunk.content, end="", flush=True)

기독교 세계관을 가지고 노아의 홍수에 대해 논의하자면, 성경의 홍수 이야기는 단순한 신화나 전설이 아니라 역사의 한 중요한 사건으로 여겨집니다. 창세기 7장에 기록된 내용은 하나님이 인류의 죄악으로 인해 세상을 심판하실 때, 노아와 그의 가족을 구원하기 위해 방주를 만들게 하셨다는 것을 전하고 있습니다. 이는 하나님의 공의와 자비를 동시에 나타내는 사건으로 해석됩니다.

성경 외에도 여러 문화에서 유사한 대홍수 이야기가 존재하는 것은 인류가 공통적으로 경험한 어떤 역사적 사건을 반영하고 있다고 볼 수 있습니다. 기독교 세계관에서는 이러한 이야기들이 서로 다른 민족과 문화에서 다르게 표현되었지만, 그 뿌리는 동일한 사건에 근거한다고 믿습니다. 즉, 노아의 홍수는 전 세계적으로 인류의 죄악과 하나님의 심판, 그리고 구원의 메시지를 전달하는 중요한 사건으로 이해됩니다.

노아의 후손들이 각기 다른 지역으로 퍼져나가고, 그 곳의 문화와 환경에 맞춰 이야기가 변형되었다는 견해는 흥미롭습니다. 이러한 과정은 하나님이 하신 일에 대한 인간의 이해와 해석이 각기 달라졌음을 보여줍니다. 그러나 성경이 기록한 사건, 즉 하나님이 세상을 심판하시고 노아에게 구원의 방주를 주신 사건은 기독교 신앙의 핵심이자, 하나님의 구원의 계획 속에서 중요한 위치를 차지하고 있습니다.

결론적으로, 기독교 관점에서 노아의 홍수는 역사적으로 실제로 있었던 사건으로 믿어지며, 이는 하나님과 인류의 관계를 드러내는 중요한 교훈과 메시지를 담고 있습니다.